# A Game of Thrones — AlphaZero training on Google Colab

This notebook clones the repository, installs the package, runs a short smoke
test and then launches a **resumable** self-play training run. Checkpoints are
written to Google Drive, so a runtime that disconnects can continue from the
last iteration.

**Before you start**

1. `Runtime > Change runtime type`: a **CPU** runtime is fine. The search is
   CPU-bound, so a GPU adds little.
2. The GitHub repository is **private**. Create a fine-grained read-only token
   for it and add it in the 🔑 panel (left sidebar) as a secret named
   `GITHUB_TOKEN`. If the repository is public, skip this.
3. Keep this tab open while training runs. If Colab disconnects you, the next
   session resumes from the last checkpoint on Drive.

The notebook installs the Python package only. If the Rust extension wheel
(`dist/agot_py-*.whl`) is installed as well, self-play and the rules engine use
it automatically; without it everything falls back to the Python engine.

In [ ]:
import os
import sys

import torch

affinity = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None
print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("usable cpus:", affinity or os.cpu_count())

In [ ]:
%cd /content
import os
from pathlib import Path

REPO_DIR = Path("/content/game-of-thrones")
if not REPO_DIR.exists():
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/canditoo2/game-of-thrones.git"
    if token:
        url = f"https://{token}@github.com/canditoo2/game-of-thrones.git"
    !git clone --depth 1 {url} {REPO_DIR}
%cd /content/game-of-thrones
!git pull --ff-only
!pip install -q -e ".[dev,rl]"

In [ ]:
!agot demo --policy greedy --seed 1
!python -m pytest tests/test_map.py tests/test_engine.py -q

## 1. Smoke gate (a few minutes)

Three short iterations first. The one thing to check is the **policy loss**: it
must not sit frozen at ~2.0 (the log of the candidate count — the signature of
the old broken policy head). A handful of iterations will not prove learning,
but a crash, a frozen loss or a broken install will show up here rather than
hours into a long run.

In [ ]:
!agot train --iterations 3 --games 12 --workers 2 --batch-games 4 --train-threads 2 --eval-games 12 --eval-every 3 --checkpoint checkpoints/smoke/agot.pt

In [ ]:
import json
from pathlib import Path

records = [
    json.loads(line)
    for line in Path("checkpoints/smoke/agot.log.jsonl").read_text().splitlines()
    if line.strip()
]
for record in records:
    print(
        f"iter {record['iteration']:>3}  "
        f"policy_loss {record['policy_loss']:.4f}  "
        f"value_loss {record['value_loss']:.4f}  "
        f"({record.get('seconds', 0):.0f}s)"
    )
losses = [record["policy_loss"] for record in records]
if losses and max(losses) - min(losses) < 0.01 and losses[0] > 1.9:
    print("WARNING: policy loss is frozen near ~2.0 — the old broken-head signature.")
else:
    print("OK: the pipeline ran and the loss is not frozen. A real run can start.")

## 2. Mount Google Drive (checkpoints must survive a disconnect)

Colab runtimes are ephemeral: when the runtime dies, `/content` is wiped. The
checkpoint, its `*.state.pt` companion and the run log go to Drive so the next
session can `--resume` exactly where the last one stopped.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

RUN_DIR = "/content/drive/MyDrive/agot/run3"
os.makedirs(RUN_DIR, exist_ok=True)
print("checkpoints ->", RUN_DIR)

## 3. Launch the training run

200 iterations at 48 games each. On a free runtime expect roughly 5 minutes per
iteration, so one session gets part-way; the cell below detects an existing
`agot.state.pt` and adds `--resume` automatically, so **re-run this cell in the
next session** to continue. Watch the printed line per iteration: by iteration
~20 the policy loss must be trending down, and every 10 iterations the run
reports a gauntlet win rate against random+greedy opponents (parity is 1/6;
the greedy reference is 23-38% depending on the field).

In [ ]:
import os

resume = os.path.exists(f"{RUN_DIR}/agot.state.pt")
print("resuming" if resume else "starting fresh")
flag = f"--resume {RUN_DIR}/agot.pt" if resume else ""

!agot train --iterations 200 --games 48 --workers 2 --batch-games 4 --opponent-rate 0.2 --train-threads 2 --temperature-rounds 3 --final-temperature 0.25 --eval-games 30 --eval-every 10 {flag} --checkpoint {RUN_DIR}/agot.pt

## 4. Monitor

The training cell prints one line per iteration. The cell below is a snapshot
you can run whenever training is not occupying the kernel: it prints the last
few iterations and renders the run log (loss curves, seconds per iteration,
and the evaluation trend against the parity and greedy lines) as a page.

In [ ]:
import json
from pathlib import Path

from IPython.display import HTML, display

from agot.training.report import render_file

log = Path(RUN_DIR) / "agot.log.jsonl"
if not log.exists():
    print("no log yet — run the training cell first")
else:
    records = [
        json.loads(line) for line in log.read_text().splitlines() if line.strip()
    ]
    for record in records[-5:]:
        rate = record.get("eval")
        eval_text = f"  eval {rate:.3f}" if rate is not None else ""
        print(
            f"iter {record['iteration']:>4}  "
            f"policy_loss {record['policy_loss']:.4f}  "
            f"value_loss {record['value_loss']:.4f}{eval_text}"
        )
    page = render_file(log, refresh=0, target_iterations=200)
    display(HTML(Path(page).read_text()))

## 5. Evaluate the best checkpoint

`agot gauntlet` puts the saved network in one rotating seat against five
opponents; `net:<path>` is a first-class policy, so the same command works for
any checkpoint. G1 is met when the win rate clears the greedy baseline with a
confidence interval that excludes it. `round-robin` compares two or more
checkpoints in a league table.

In [ ]:
!agot gauntlet --candidate net:{RUN_DIR}/agot-best.pt --opponents random --games 60

In [ ]:
!agot round-robin --checkpoints {RUN_DIR}/agot.pt {RUN_DIR}/agot-best.pt --games 30

## Troubleshooting

- **Out of memory** (a free runtime has ~12 GB): lower the replay buffer,
  `--buffer 100000`, and/or `--games 32`.
- **Disconnected**: re-run the clone/install cell, the Drive cell and the
  launch cell; it resumes from the Drive checkpoint. The replay buffer
  restarts empty by design, so a transient loss spike on the first resumed
  iteration is expected.
- **Flat policy loss for many iterations**: stop and read `docs/methods.md` —
  that is the old broken-head signature, not convergence.
- **`agot: command not found`**: re-run the install cell.
- **More throughput**: `--workers 4` if the runtime has the cores; the printed
  per-iteration time is the number to watch.